In [0]:
# bronze/03_coin_universe.py
# Tracks daily universe membership — which coins were in top-N on each date

from pyspark.sql import functions as F
from datetime import datetime, timezone

DB_NAME        = "crypto_db"
BRONZE_MARKET  = f"{DB_NAME}.bronze_market_data"
COIN_UNIVERSE  = f"{DB_NAME}.coin_universe"
DIM_COIN       = f"{DB_NAME}.dim_coin"

# Always use UTC date (string format matches bronze)
RUN_DT = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# Standard column order (VERY IMPORTANT for union)
COLS = [
    "coin_id",
    "name",
    "symbol",
    "market_cap_rank",
    "universe_date",
    "is_active",
    "entry_reason"
]

# ─────────────────────────────────────────────────────────────
# 1. TODAY'S ACTIVE UNIVERSE (Top-N coins today)
# ─────────────────────────────────────────────────────────────

df_today = (
    spark.table(BRONZE_MARKET)
    .filter(F.col("ingestion_date") == RUN_DT)
    .select(
        F.col("id").alias("coin_id"),
        "name",
        "symbol",
        "market_cap_rank",
        F.col("ingestion_date").alias("universe_date"),
    )
    .withColumn("is_active",    F.lit(True))
    .withColumn("entry_reason", F.lit("in_top_n"))
    .select(COLS)   # enforce order
)

# ─────────────────────────────────────────────────────────────
# 2. FIND EXITED COINS (present yesterday, not today)
# ─────────────────────────────────────────────────────────────

if spark.catalog.tableExists(COIN_UNIVERSE):

    yesterday = (
        spark.table(COIN_UNIVERSE)
        .filter(F.col("universe_date") == F.date_sub(F.lit(RUN_DT), 1))
        .filter(F.col("is_active") == True)
        .select("coin_id")
    )

    # Coins that disappeared today
    exited = (
        yesterday
        .join(df_today.select("coin_id"), on="coin_id", how="left_anti")
        .withColumn("universe_date", F.lit(RUN_DT))
        .withColumn("is_active",     F.lit(False))
        .withColumn("entry_reason",  F.lit("exited_top_n"))
    )

    # Enrich exited coins with metadata
    exited = (
        exited
        .join(
            spark.table(DIM_COIN).select(
                F.col("id").alias("coin_id"),
                "name",
                "symbol",
                F.col("market_cap_rank")
            ),
            on="coin_id",
            how="left"
        )
        .select(COLS)   # enforce same order
    )

    # 🔥 CRITICAL FIX → unionByName (NOT union)
    df_today = df_today.unionByName(exited, allowMissingColumns=True)

# ─────────────────────────────────────────────────────────────
# 3. WRITE USING MERGE (idempotent)
# ─────────────────────────────────────────────────────────────

df_today.createOrReplaceTempView("_stg_universe")

if not spark.catalog.tableExists(COIN_UNIVERSE):

    (
        df_today.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("universe_date")
        .saveAsTable(COIN_UNIVERSE)
    )

    print(f"Created {COIN_UNIVERSE}")

else:

    spark.sql(f"""
        MERGE INTO {COIN_UNIVERSE} tgt
        USING _stg_universe src
        ON  tgt.coin_id       = src.coin_id
        AND tgt.universe_date = src.universe_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"Merged {COIN_UNIVERSE}")

# ─────────────────────────────────────────────────────────────
# 4. SUMMARY (validation)
# ─────────────────────────────────────────────────────────────

print("\n=== Universe summary today ===")

(
    spark.table(COIN_UNIVERSE)
    .filter(F.col("universe_date") == RUN_DT)
    .groupBy("is_active", "entry_reason")
    .count()
    .show()
)

In [0]:
%sql
DESCRIBE TABLE crypto_db.coin_universe;